In [1]:
from datasets import load_dataset, Dataset
from sentence_transformers import SentenceTransformer
from sentence_transformers.sentence_transformer import losses
from sentence_transformers.sentence_transformer.evaluation import EmbeddingSimilarityEvaluator
from sentence_transformers.sentence_transformer.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer.trainer import SentenceTransformerTrainer

# 加载训练数据
train_dataset = load_dataset(
    "nyu-mll/glue", "mnli", split="train"
).select(range(50_000)).remove_columns("idx")

# 中性/矛盾 = 0, 蕴含=1
mapping = {2: 0, 1:0, 0: 1}
train_dataset = Dataset.from_dict({
    "sentence1": train_dataset["premise"],
    "sentence2": train_dataset["hypothesis"],
    "label": [float(mapping[label]) for label in train_dataset["label"]]
})

# 选择基座模型
embedding_model = SentenceTransformer('bert-base-uncased', device="cuda")

# 定义损失函数
train_loss = losses.CosineSimilarityLoss(model=embedding_model)

# 定义评估器，使用语义文本相似度基准(Semantic Textual Similarity Benchmark, STSB)
# 这是一个由人工标注的句子对数据集，相似度分数在 1 ~ 5 之间
val_sts = load_dataset("nyu-mll/glue", "stsb", split="validation")
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]], # 值转换为 0~1 之间
    main_similarity="cosine"
)

# 定义训练参数
args = SentenceTransformerTrainingArguments(
    output_dir="cosineloss_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100
)

# 训练模型
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)

print(evaluator(embedding_model))
trainer.train()
print(evaluator(embedding_model))

embedding_model.save("cosineloss_embedding_model")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

{'pearson_cosine': 0.5917194531209226, 'spearman_cosine': 0.5931742011707938}


Step,Training Loss
100,0.230911
200,0.168756
300,0.169068
400,0.156070
500,0.153544
600,0.155934
700,0.149661
800,0.156906
900,0.147914
1000,0.146370


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'pearson_cosine': 0.7366206129116531, 'spearman_cosine': 0.7381769311319638}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [3]:
evaluator(embedding_model)

{'pearson_cosine': 0.7250524707965853, 'spearman_cosine': 0.7289073126352569}